In [18]:
import os
import re
import pandas as pd

repo_dir = "/Users/haya1/Documents/LanguageModel_Labels/congressional_bills"
os.chdir(repo_dir)

import nltk
nltk.download('stopwords')
nltk.download('wordnet')

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_curve, auc

[nltk_data] Downloading package stopwords to /Users/haya1/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/haya1/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


## Data Preprocessing

In [19]:
SEED = 123
STOPWORDS = nltk.corpus.stopwords.words('english')
LEMMATIZER = nltk.stem.WordNetLemmatizer()
VECTORIZER = CountVectorizer()

In [20]:
def clean_text(X):    
    # lowecase and remove punctuation
    X = re.sub(r'[^a-zA-Z]', ' ', X.lower())

    # remove stopwords 
    words = [word for word in X.split() if word not in set(STOPWORDS)]

    # stemming (maybe omit and see if results change?)
    words = [LEMMATIZER.lemmatize(word) for word in words]
    
    # join back into string and return (sklearn vectorizer wants string as input)
    return ' '.join(words)

def bag_of_words(X):
    # Create BOW representation    
    return VECTORIZER.fit_transform(X.apply(clean_text))

In [17]:
bills_prompts_responses = pd.read_csv(os.path.join(repo_dir, f"02_llm/bills_prompts_responses_10000.csv"))
X = bag_of_words(bills_prompts_responses["Description"])
# print(X[0])
y = bills_prompts_responses["Major"] !=  bills_prompts_responses["MajorLLM"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=SEED)

## Logistic Regression

### Lasso

In [13]:
# Predict
lasso_model = LogisticRegression(penalty='l1', solver='liblinear', max_iter=1000)
lasso_model.fit(X_train, y_train)

# Evaluate 
y_pred_lasso = lasso_model.predict(X_test)
y_probs_lasso = lasso_model.predict_proba(X_test)[:,1]
fpr_lasso, tpr_lasso, thresholds_lasso = roc_curve(y_test, y_probs_lasso)
auc_score_lasso = auc(fpr_lasso, tpr_lasso)
lasso_auc.append(auc_score_lasso)
print("Lasso Accuracy:", accuracy_score(y_test, y_pred_lasso))
print("AUC:", auc_score_lasso)
print("Lasso Classification Report:\n", classification_report(y_test, y_pred_lasso))

LogisticRegression(max_iter=1000, penalty='l1', solver='liblinear')

In [ ]:
# Logistic Regression with ridge (using instead of sklearn canned Ridge classifer)
ridge_model = LogisticRegression(penalty='l2', solver='liblinear', max_iter=1000)
ridge_model.fit(X_train, y_train)

# Evaluate ridge
y_pred_ridge = ridge_model.predict(X_test)
y_probs_ridge = ridge_model.predict_proba(X_test)[:,1]

fpr_ridge, tpr_ridge, thresholds_lasso = roc_curve(y_test, y_probs_ridge)
auc_score_ridge = auc(fpr_ridge, tpr_ridge)
ridge_auc.append(auc_score_ridge)
print("Ridge Accuracy:", accuracy_score(y_test, y_pred_ridge))
print("AUC:", auc_score_ridge)
print("Ridge Classification Report:\n", classification_report(y_test, y_pred_ridge))

In [ ]:
# plot auc
plt.figure()
roc_plot(tpr_lasso, fpr_lasso, auc_score=auc_score_lasso, model_name="log reg w/ lasso", dataset = dataset, model = model)
roc_plot(tpr_ridge, fpr_ridge, auc_score=auc_score_ridge, model_name="log reg w/ ridge", dataset = dataset, model = model)
plt.savefig(f"../figures/{dataset}/{model}.png", bbox_inches = "tight")